# 01 - Verifikasi data FLEURS

Notebook ini memeriksa apakah audio FLEURS Indonesia dan Inggris lengkap, bisa dibaca, sesuai dengan daftar transkrip, dan punya duplikat. Jalankan semua sel untuk melihat hasilnya.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import re
import statistics
import numpy as np
from scipy.io import wavfile
from collections import Counter, defaultdict

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
DATA_ROOT = root / "data"
REPORT_PATH = root / "results/data_audit.json"

LANGUAGES = ("id_id", "en_us")
SPLITS = ("train", "dev", "test")
MIN_SECONDS, MAX_SECONDS = 0.25, 30.0  # Tanda untuk ditinjau, bukan filter otomatis.
print("Sumber data:", DATA_ROOT)
print("Tujuan laporan:", REPORT_PATH)

In [ ]:
def inspect_wav(path):
    rate, samples = wavfile.read(path)
    if rate <= 0 or samples.size == 0 or samples.ndim not in (1, 2):
        raise ValueError("sample rate atau bentuk audio tidak valid")
    if not np.isfinite(samples).all():
        raise ValueError("audio mengandung NaN atau infinity")

    channels = 1 if samples.ndim == 1 else samples.shape[1]
    canonical = samples.astype(samples.dtype.newbyteorder("<"), copy=False)
    digest = hashlib.sha256()
    digest.update(f"{rate}/{channels}/{canonical.dtype.str}:".encode("ascii"))
    digest.update(canonical.tobytes())
    return len(samples) / rate, rate, channels, digest.hexdigest()


def audit_fleurs(root):
    report = {"data_root": str(root), "splits": {}, "issues": [], "overlaps": {}}
    issues = report["issues"]
    groups = {name: defaultdict(list) for name in ("sentence_ids", "pcm_sha256", "normalized_transcripts")}

    for language in LANGUAGES:
        for split in SPLITS:
            key = f"{language}/{split}"
            metadata = root / language / f"{split}.tsv"
            audio_dir = root / language / "audio" / split
            summary = {"metadata_rows": None, "wav_files": None, "readable_pairs": 0}
            report["splits"][key] = summary
            referenced = set()
            durations, rates, channels = [], Counter(), Counter()
            available = {p.name for p in audio_dir.iterdir() if p.is_file() and p.suffix.lower() == ".wav"} if audio_dir.is_dir() else set()
            if audio_dir.is_dir():
                summary["wav_files"] = len(available)
            else:
                issues.append({"split": key, "type": "missing_audio_dir", "path": str(audio_dir)})
            if not metadata.is_file():
                issues.append({"split": key, "type": "missing_metadata", "path": str(metadata)})
                print(f"{key}: TSV belum ada, pembacaan audio dilewati.", flush=True)
                continue

            summary["metadata_rows"] = 0
            with metadata.open(encoding="utf-8-sig", newline="") as stream:
                for line, row in enumerate(csv.reader(stream, delimiter=chr(9), quoting=csv.QUOTE_NONE), 1):
                    if not any(field.strip() for field in row):
                        continue
                    summary["metadata_rows"] += 1
                    entry = {"split": key, "row": line}
                    if len(row) != 7 or not row[0].isdigit():
                        issues.append({**entry, "type": "invalid_metadata_row"})
                        continue
                    sentence_id, filename, raw_text = row[:3]
                    if Path(filename).name != filename or "/" in filename or chr(92) in filename or not filename.lower().endswith(".wav"):
                        issues.append({**entry, "type": "invalid_audio_filename"})
                        continue
                    entry = {**entry, "audio": filename}
                    if filename in referenced:
                        issues.append({**entry, "type": "duplicate_metadata_audio"})
                    referenced.add(filename)
                    groups["sentence_ids"][sentence_id].append(entry)
                    normalized = " ".join(re.findall(r"[^\W_]+", raw_text.casefold()))
                    if normalized:
                        groups["normalized_transcripts"][(language, normalized)].append(entry)
                    else:
                        issues.append({**entry, "type": "empty_transcript"})
                    if filename not in available:
                        if audio_dir.is_dir():
                            issues.append({**entry, "type": "missing_wav"})
                        continue
                    try:
                        duration, rate, channel_count, digest = inspect_wav(audio_dir / filename)
                    except (OSError, EOFError, ValueError) as exc:
                        issues.append({**entry, "type": "unreadable_wav", "detail": str(exc)})
                        continue
                    summary["readable_pairs"] += bool(normalized)
                    durations.append(duration)
                    rates[rate] += 1
                    channels[channel_count] += 1
                    groups["pcm_sha256"][digest].append(entry)
                    if not MIN_SECONDS <= duration <= MAX_SECONDS:
                        issues.append({**entry, "type": "review_duration", "seconds": round(duration, 3)})
            if summary["metadata_rows"] == 0:
                issues.append({"split": key, "type": "empty_metadata"})
            for filename in sorted(available - referenced):
                issues.append({"split": key, "type": "unreferenced_wav", "audio": filename})
            summary.update(
                hours=round(sum(durations) / 3600, 3),
                min_seconds=round(min(durations), 3) if durations else None,
                median_seconds=round(statistics.median(durations), 3) if durations else None,
                max_seconds=round(max(durations), 3) if durations else None,
                sample_rates=dict(rates),
                channel_counts=dict(channels),
            )
            print(f"{key}: selesai, {summary['readable_pairs']} dari {summary['metadata_rows']} pasangan terbaca.", flush=True)

    for language in LANGUAGES:
        summaries = [report["splits"][f"{language}/{split}"] for split in SPLITS]
        counts = [summary["metadata_rows"] for summary in summaries]
        total = sum(counts) if all(count is not None for count in counts) else 0
        for summary in summaries:
            summary["metadata_percent"] = round(summary["metadata_rows"] / total * 100, 1) if total else None

    for name, matches in groups.items():
        report["overlaps"][name] = [
            entries for entries in matches.values()
            if len({entry["split"].rsplit("/", 1)[1] for entry in entries}) > 1
        ]
    report["duplicate_pcm_within_split"] = [
        entries for entries in groups["pcm_sha256"].values()
        if len(entries) > 1 and len({entry["split"] for entry in entries}) == 1
    ]
    return report

In [ ]:
from IPython.display import Markdown, display

ISSUE_GUIDE = {
    'missing_audio_dir': ('Folder WAV belum ada', True),
    'missing_metadata': ('Metadata TSV belum ada', True),
    'empty_metadata': ('Metadata kosong', True),
    'invalid_metadata_row': ('Baris TSV tidak valid', True),
    'invalid_audio_filename': ('Nama WAV tidak valid', True),
    'duplicate_metadata_audio': ('Satu WAV tercatat berkali-kali', True),
    'empty_transcript': ('Transkrip kosong', True),
    'missing_wav': ('WAV yang tercatat tidak ditemukan', True),
    'unreadable_wav': ('WAV gagal dibaca', True),
    'unreferenced_wav': ('WAV tanpa baris metadata', False),
    'review_duration': ('Durasi di luar batas tinjauan', False),
}
OVERLAP_GUIDE = {
    "sentence_ids": "ID kalimat yang sama muncul di split berbeda",
    "pcm_sha256": "Audio dengan sampel digital identik muncul di split berbeda",
    "normalized_transcripts": "Teks yang sama untuk satu bahasa muncul di split berbeda",
    "within_split": "Audio dengan sampel digital identik berulang dalam split yang sama",
}


def format_number(value, decimals=0):
    if value is None:
        return "Belum tersedia"
    return f"{value:,.{decimals}f}".translate(str.maketrans(",.", ".,"))


def example_label(entry):
    location = entry["split"]
    name = entry.get("audio") or entry.get("path") or ""
    row = f", baris TSV {entry['row']}" if "row" in entry else ""
    return f"{location}{row}" + (f" — {name}" if name else "")


def human_report(report):
    issues = defaultdict(list)
    for issue in report["issues"]:
        issues[issue["type"]].append(issue)
    overlaps = {**report["overlaps"], "within_split": report["duplicate_pcm_within_split"]}
    incomplete = any(not part["metadata_rows"] or part["readable_pairs"] != part["metadata_rows"]
                     for part in report["splits"].values())
    if incomplete or any(ISSUE_GUIDE[k][1] for k in issues):
        verdict = "Ada pasangan audio dan transkrip yang belum valid."
    elif issues or any(overlaps.values()):
        verdict = "Semua pasangan terbaca. Ada catatan di bawah."
    else:
        verdict = "Semua pasangan terbaca."

    lines = ["## Hasil audit FLEURS", "", f"**{verdict}**", "",
             "| Bahasa / split | Baris TSV | File WAV | Pasangan terbaca | Total audio | Durasi tengah | Sample rate / kanal |",
             "| --- | ---: | ---: | ---: | ---: | ---: | --- |"]
    for key, summary in report["splits"].items():
        rates = ", ".join(f"{format_number(rate)} Hz" for rate in sorted(summary.get("sample_rates", {})))
        channels = ", ".join("mono" if count == 1 else "stereo" if count == 2 else f"{count} kanal"
                             for count in sorted(summary.get("channel_counts", {})))
        audio_format = f"{rates} / {channels}" if rates else "Belum terbaca"
        hours = f"{format_number(summary['hours'], 2)} jam" if rates else "Belum terbaca"
        median = f"{format_number(summary['median_seconds'], 1)} detik" if rates else "Belum terbaca"
        lines.append(f"| {key} | {format_number(summary['metadata_rows'])} | {format_number(summary['wav_files'])} | "
                     f"{format_number(summary['readable_pairs'])} | {hours} | {median} | {audio_format} |")

    lines.append("")
    for kind, entries in issues.items():
        label, _ = ISSUE_GUIDE[kind]
        lines.append(f"- {label}: **{format_number(len(entries))}**.")
        for entry in entries[:3]:
            detail = entry.get("detail", "")
            if "seconds" in entry:
                detail = f"{format_number(entry['seconds'], 2)} detik"
            lines.append(f"  - {example_label(entry)}" + (f" ({detail})" if detail else ""))

    for key, groups in overlaps.items():
        if groups:
            lines.append(f"- {OVERLAP_GUIDE[key]}: **{format_number(len(groups))} kelompok**.")
            for group in groups[:2]:
                lines.append("  - " + ", ".join(example_label(entry) for entry in group[:3]))
    if not issues and not any(overlaps.values()):
        lines.append("Tidak ada masalah atau duplikat yang terdeteksi.")
    return "\n".join(lines)


audit = audit_fleurs(DATA_ROOT)
summary = human_report(audit)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(audit, indent=2, ensure_ascii=False) + chr(10), encoding="utf-8")
readable_path = REPORT_PATH.with_suffix(".md")
readable_path.write_text(summary + chr(10), encoding="utf-8")
display(Markdown(summary))
print(f"Laporan yang bisa dibaca: {readable_path}")